In [ ]:
import numpy as np 
import pandas pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


df = pd.read_csv('creditcard.csv')

normal_df = df[df['Class']==0]
fraud_df = df[df['Class']==1]

normal_features = normal_df.drop('Class',axis=1)
fraud_features = fraud_df.drop('Class',axis=1)

Scaler = StandardScaler()
normal_scaled = Scaler.fit_transform(normal_features)
fraud_scaled = Scaler.transform(fraud_features)

X_train, X_test = train_test_split(normal_scaled, test_size=0.2, random_state = 42)

input_dim = X_train.shape[1]
model = keras.Sequential([
    layers.Dense(16, activation='relu',input_shape=(input_dim,)),
    layers.Dense(8, activation='relu'),

    layers.Dense(16, activation='relu'),
    layers.Dense(input_dim, activation='linear')
])


model.compile(optimizer='adam',loss='mse',metrics=['mae'])

H = model.fit(X_train,X_train, epochs=10, validation_data=(X_test,X_test))

reconstructions = model.predict(fraud_scaled)

mse = np.mean(np.power(fraud_scaled - reconstructions, 2),axis=1)

threshold = np.percentile(mse,99)

anomalies = mse > threshold

print(f"Total Frauds : {sum(anomalies)}")